# LeetCode 427 — Construct Quad Tree bằng Python

Link đề: https://leetcode.com/problems/construct-quad-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Array, Divide and Conquer (chia để trị), Matrix

Skeleton LeetCode:

```python
"""
# Definition for a QuadTree node.
class Node:
    def __init__(self, val, isLeaf, topLeft, topRight, bottomLeft, bottomRight):
        self.val = val
        self.isLeaf = isLeaf
        self.topLeft = topLeft
        self.topRight = topRight
        self.bottomLeft = bottomLeft
        self.bottomRight = bottomRight
"""

class Solution:
    def construct(self, grid: List[List[int]]) -> 'Node':
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án đã được kiểm chứng bằng code (dựng lại lưới từ cây, đối chiếu bốn cách trên hàng trăm lưới ngẫu nhiên). Phần giải thích **định dạng tuần tự hóa** (mục 3.5) mình rút ra từ hai ví dụ của đề — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc. Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: biểu diễn một lưới vuông `n × n` chỉ gồm `0/1` bằng **cây tứ phân (quad tree)**: vùng đồng nhất thì là **nút lá**, ngược lại chia **4 góc phần tư** và đệ quy.
2. Nắm các công cụ: lớp `Node` có 6 thuộc tính, chia lưới bằng **toạ độ góc + độ dài cạnh** `(r, c, size)`, kiểm tra "cả vùng toàn một giá trị" bằng `all(...)`/`set`, **tổng tiền tố 2D (prefix sum)**, hàng đợi để **tuần tự hóa** cây, và hàm **dựng lại lưới từ cây** để tự kiểm chứng.
3. Đi từ **chia để trị kèm quét vùng** (baseline) → **quét vùng trong `O(1)` bằng prefix sum** → **dựng từ dưới lên rồi gộp 4 con giống nhau** → **gộp từng tầng bằng vòng lặp (không đệ quy)**.
4. Hiểu mấu chốt: vùng `size × size` chia thành 4 vùng `size/2 × size/2` (vì `n` là luỹ thừa của 2 nên luôn chia đều), và độ sâu chỉ tối đa `log₂ n ≤ 6`.
5. Tránh các lỗi kinh điển: quên cộng **offset** `(r, c)` khi chia góc, chỉ kiểm tra vài ô thay vì cả vùng, quên **gộp** 4 lá giống nhau (cây không tối giản), thiếu trường hợp cơ sở `size = 1`.

## 2. Đề bài & ví dụ

Cho lưới `grid` kích thước `n × n` chỉ gồm `0` và `1`. Hãy dựng **quad tree** biểu diễn `grid` và trả về gốc.

Mỗi nút có hai thuộc tính:

- `val`: `True` nếu vùng đại diện toàn `1`, `False` nếu toàn `0`. Với nút **không phải lá**, `val` có thể là **bất kỳ** (cả hai đều được chấp nhận).
- `isLeaf`: `True` nếu là lá, `False` nếu có đủ bốn con: `topLeft`, `topRight`, `bottomLeft`, `bottomRight`.

Cách dựng:

1. Nếu vùng hiện tại **toàn cùng giá trị** → nút lá (`isLeaf = True`, `val` = giá trị đó, bốn con là `None`), dừng.
2. Ngược lại → `isLeaf = False`, chia vùng thành **bốn góc phần tư** bằng nhau và đệ quy cho từng góc.

**Ví dụ 1:** `grid = [[0,1],[1,0]]` → `[[0,1],[1,0],[1,1],[1,1],[1,0]]`. Gốc là `[0,1]` (không lá, `val` tuỳ ý — ở đây `1`), rồi bốn lá: trái-trên `0`, phải-trên `1`, trái-dưới `1`, phải-dưới `0`.

**Ví dụ 2:** lưới `8 × 8` (cho ở dưới) → `[[0,1],[1,1],[0,1],[1,1],[1,0],null,null,null,null,[1,0],[1,0],[1,1],[1,1]]`.

**Ràng buộc:** `n == grid.length == grid[i].length`; `n = 2^x` với `0 <= x <= 6` (tức `n ∈ {1,2,4,8,16,32,64}`).

**Định dạng kết quả** `[isLeaf, val]` mỗi nút theo thứ tự từng tầng, `null` là chỗ không có nút; cách đọc chi tiết ở mục 3.5.

In [1]:
grid_ex1 = [[0, 1], [1, 0]]
grid_ex2 = [
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 0],
]
exp_ex1 = [[0, 1], [1, 0], [1, 1], [1, 1], [1, 0]]
exp_ex2 = [[0, 1], [1, 1], [0, 1], [1, 1], [1, 0], None, None, None, None, [1, 0], [1, 0], [1, 1], [1, 1]]
for g in (grid_ex1, grid_ex2):
    print(len(g), "x", len(g[0]))

2 x 2
8 x 8


## 3. Kiến thức nền

### 3.1. Nút `Node` của quad tree

Mỗi nút có **6 thuộc tính**: `val`, `isLeaf` và bốn con. Lá thì bốn con đều `None`. Trong notebook ta đặt giá trị mặc định cho tiện, nhưng **bài nộp luôn truyền đủ cả 6 tham số** cho chắc.

In [2]:
class Node:
    def __init__(self, val=False, isLeaf=False, topLeft=None, topRight=None, bottomLeft=None, bottomRight=None):
        self.val = val
        self.isLeaf = isLeaf
        self.topLeft = topLeft
        self.topRight = topRight
        self.bottomLeft = bottomLeft
        self.bottomRight = bottomRight


leaf1 = Node(True, True, None, None, None, None)
leaf0 = Node(False, True, None, None, None, None)
root = Node(True, False, leaf0, leaf1, leaf1, leaf0)      # lưới [[0,1],[1,0]]
print(root.isLeaf, root.topLeft.val, root.topRight.val, root.bottomLeft.val, root.bottomRight.val)

False False True True False


### 3.2. Chia lưới bằng `(r, c, size)` — góc trên-trái và độ dài cạnh

Thay vì cắt (copy) lưới con, ta chỉ nhớ **vùng** bằng 3 số: hàng `r`, cột `c` của ô **trên-trái**, và cạnh `size`. Vùng chứa các ô `grid[i][j]` với `r <= i < r+size`, `c <= j < c+size`. Chia đôi: `h = size // 2`.

| Góc | `(r', c')` | cạnh |
|---|---|---|
| trên-trái | `(r, c)` | `h` |
| trên-phải | `(r, c + h)` | `h` |
| dưới-trái | `(r + h, c)` | `h` |
| dưới-phải | `(r + h, c + h)` | `h` |

**Nhớ cộng offset** `r`, `c` (đừng chỉ dùng `h`) — vùng con không phải lúc nào cũng nằm góc `(0,0)`!

In [3]:
def quadrants(r, c, size):
    h = size // 2
    return {
        "trên-trái": (r, c, h),
        "trên-phải": (r, c + h, h),
        "dưới-trái": (r + h, c, h),
        "dưới-phải": (r + h, c + h, h),
    }

print(quadrants(0, 0, 8))
print(quadrants(4, 4, 4))     # vùng dưới-phải của lưới 8x8, lại chia tiếp

{'trên-trái': (0, 0, 4), 'trên-phải': (0, 4, 4), 'dưới-trái': (4, 0, 4), 'dưới-phải': (4, 4, 4)}
{'trên-trái': (4, 4, 2), 'trên-phải': (4, 6, 2), 'dưới-trái': (6, 4, 2), 'dưới-phải': (6, 6, 2)}


### 3.3. Kiểm tra "cả vùng toàn một giá trị"

Với vùng `(r, c, size)`, lấy ô đầu `first = grid[r][c]` và kiểm tra **mọi** ô bằng `first`. Dùng `all(...)` với biểu thức sinh (generator): nó **dừng ngay khi gặp ô khác** (short-circuit). Cách khác: gom giá trị vào `set`, nếu `len(set) == 1` thì đồng nhất.

In [4]:
def uniform(grid, r, c, size):
    first = grid[r][c]
    return all(grid[i][j] == first for i in range(r, r + size) for j in range(c, c + size))

g = [[1, 1, 0, 0],
     [1, 1, 0, 1],
     [0, 0, 1, 1],
     [0, 0, 1, 1]]
print(uniform(g, 0, 0, 2), uniform(g, 0, 2, 2), uniform(g, 2, 0, 2), uniform(g, 0, 0, 4))
# True False True False

print({g[i][j] for i in range(0, 2) for j in range(2, 4)})   # {0, 1}: không đồng nhất

True False True False
{0, 1}


### 3.4. Tổng tiền tố 2D (prefix sum)

`S[i][j]` = tổng các ô trong hình chữ nhật từ `(0,0)` đến `(i-1,j-1)`. Dựng bằng

`S[i+1][j+1] = S[i][j+1] + S[i+1][j] - S[i][j] + grid[i][j]`

(cộng phần trên và phần trái, trừ phần giao bị cộng hai lần). Khi đó tổng một vùng `(r, c, size)` chỉ cần **4 phép tra bảng**:

`sum = S[r+size][c+size] - S[r][c+size] - S[r+size][c] + S[r][c]`

Vùng toàn `0` ⇔ tổng `= 0`; toàn `1` ⇔ tổng `= size*size`.

In [5]:
def build_prefix(grid):
    n = len(grid)
    S = [[0] * (n + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(n):
            S[i + 1][j + 1] = S[i][j + 1] + S[i + 1][j] - S[i][j] + grid[i][j]
    return S

def region_sum(S, r, c, size):
    return S[r + size][c + size] - S[r][c + size] - S[r + size][c] + S[r][c]

S = build_prefix(g)
print(region_sum(S, 0, 0, 2), region_sum(S, 0, 2, 2), region_sum(S, 2, 2, 2), region_sum(S, 0, 0, 4))   # 4 1 4 8

4 1 4 9


### 3.5. Tuần tự hóa (serialize): đọc kết quả của LeetCode

Kết quả là list theo **thứ tự từng tầng**. Mỗi nút là `[isLeaf, val]` (viết `1/0`). Quy tắc rút ra từ hai ví dụ: lấy lần lượt từng nút từ **hàng đợi**; với mỗi nút ta ghi **4 ô con** theo thứ tự `topLeft, topRight, bottomLeft, bottomRight` — có thì ghi `[isLeaf, val]` và xếp vào hàng đợi, không có thì ghi `null`. Cuối cùng **cắt các `null` thừa ở cuối**.

Hệ quả: nút **lá** sinh ra 4 `null` (con không có); trong Ví dụ 2 ta thấy đúng `4` `null` đứng sau 4 nút tầng hai, là con của lá `topLeft`(tầng hai), còn nút `topRight` (không lá) mới có 4 con thật. Hàm dưới đây sẽ cho đúng hai ví dụ của đề.

In [6]:
from collections import deque

def serialize(root):
    if root is None:
        return []
    out = [[int(root.isLeaf), int(root.val)]]
    q = deque([root])
    while q:
        node = q.popleft()
        for child in (node.topLeft, node.topRight, node.bottomLeft, node.bottomRight):
            if child is None:
                out.append(None)
            else:
                out.append([int(child.isLeaf), int(child.val)])
                q.append(child)
    while out and out[-1] is None:
        out.pop()
    return out

print(serialize(root))                      # [[0, 1], [1, 0], [1, 1], [1, 1], [1, 0]]
assert serialize(root) == exp_ex1

[[0, 1], [1, 0], [1, 1], [1, 1], [1, 0]]


### 3.6. Giám khảo: dựng lại lưới từ cây và kiểm tra cây tối giản

Vì **`val` của nút không lá là tuỳ ý**, ta không so sánh từng chữ `val`. Thay vào đó viết một **giám khảo độc lập** `check_quadtree(root, grid)`:

1. **Dựng lại lưới** từ cây (`decode_grid`) và so với `grid` gốc.
2. Mỗi **lá** phải có 4 con `None`; mỗi **nút không lá** phải có đủ 4 con.
3. Cây phải **tối giản** đúng theo quy tắc đề: không có nút không lá nào mà **cả 4 con đều là lá cùng giá trị** (vì khi đó vùng đó đồng nhất, lẽ ra phải là một lá).
4. Lá chỉ được ở vùng đồng nhất (suy ra từ 1 do dựng lại đúng), và không lá thì vùng không đồng nhất (suy ra từ 3 bằng quy nạp từ dưới lên).

In [7]:
def decode_grid(root, n):
    grid = [[None] * n for _ in range(n)]

    def fill(node, r, c, size):
        if node.isLeaf:
            for i in range(r, r + size):
                for j in range(c, c + size):
                    grid[i][j] = int(node.val)
            return
        h = size // 2
        fill(node.topLeft, r, c, h)
        fill(node.topRight, r, c + h, h)
        fill(node.bottomLeft, r + h, c, h)
        fill(node.bottomRight, r + h, c + h, h)

    fill(root, 0, 0, n)
    return grid


def check_quadtree(root, grid):
    n = len(grid)
    if root is None:
        return False, "gốc là None"

    def structure(node, size):
        kids = (node.topLeft, node.topRight, node.bottomLeft, node.bottomRight)
        if node.isLeaf:
            if any(k is not None for k in kids):
                return "lá nhưng có con"
            return None
        if size == 1:
            return "nút không lá ở vùng 1x1"
        if any(k is None for k in kids):
            return "không lá nhưng thiếu con"
        if all(k.isLeaf for k in kids) and len({bool(k.val) for k in kids}) == 1:
            return "chưa gộp: 4 lá cùng giá trị dưới một nút không lá"
        for k in kids:
            m = structure(k, size // 2)
            if m:
                return m
        return None

    msg = structure(root, n)
    if msg:
        return False, msg
    if decode_grid(root, n) != grid:
        return False, "lưới dựng lại khác lưới gốc"
    return True, "ok"


def count_nodes(node):
    if node is None:
        return 0
    return 1 + sum(count_nodes(k) for k in (node.topLeft, node.topRight, node.bottomLeft, node.bottomRight))

print(check_quadtree(root, grid_ex1))      # (True, 'ok')
print(check_quadtree(root, [[0, 1], [1, 1]]))

(True, 'ok')
(False, 'lưới dựng lại khác lưới gốc')


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lưới `4 × 4` (tự dựng):

```
1 1 | 0 0
1 1 | 0 1
----+----
0 0 | 1 1
0 0 | 1 1
```

Làm tay từ gốc:

1. Cả lưới `4×4` có cả `0` và `1` → **không lá**, chia bốn góc `2×2`.
2. **Trên-trái** `[[1,1],[1,1]]` toàn `1` → **lá** `1`.
3. **Trên-phải** `[[0,0],[0,1]]` có cả hai → **không lá**, chia thành 4 ô `1×1`: `0, 0, 0, 1` → bốn lá.
4. **Dưới-trái** toàn `0` → lá `0`. **Dưới-phải** toàn `1` → lá `1`.

Tổng: `1 (gốc) + 4 (góc) + 4 (ô 1×1 trong trên-phải) = 9` nút. Hai điều quan trọng cần nhớ:

- **Ô `1×1` luôn đồng nhất** ⇒ đệ quy chắc chắn dừng ở `size = 1` (trường hợp cơ sở).
- Nếu cả lưới đồng nhất ngay từ đầu thì cây **chỉ có một nút lá**.

## 5. Cách 1 — Brute Force / baseline: quét từng vùng, rồi chia nếu chưa đồng nhất

**Ý tưởng.** Làm đúng theo đề: với vùng `(r, c, size)`:

1. Quét **mọi ô** của vùng (`uniform`). Nếu đồng nhất → trả về **lá** có `val = grid[r][c]`.
2. Ngược lại → chia 4 góc (mục 3.2), đệ quy, trả về nút **không lá** (`val = True` tuỳ ý) với 4 con.

Không cần trường hợp cơ sở riêng cho `size = 1`: vùng một ô luôn đồng nhất nên rơi vào bước 1.

In [8]:
def construct_scan(grid):
    def dfs(r, c, size):
        if uniform(grid, r, c, size):
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        return Node(True, False,
                    dfs(r, c, h),
                    dfs(r, c + h, h),
                    dfs(r + h, c, h),
                    dfs(r + h, c + h, h))

    return dfs(0, 0, len(grid))


r1 = construct_scan(grid_ex1)
r2 = construct_scan(grid_ex2)
print(serialize(r1) == exp_ex1, serialize(r2) == exp_ex2)
print(check_quadtree(r1, grid_ex1), check_quadtree(r2, grid_ex2))

True True
(True, 'ok') (True, 'ok')


**Độ phức tạp (Cách 1).** Gọi `n` là cạnh lưới (`N = n²` ô). Ở mỗi **tầng** của cây các vùng không giao nhau và phủ cả lưới, nên mỗi tầng quét tối đa `n²` ô. Có tối đa `log₂ n + 1` tầng ⇒ Time **`O(n² log n)`**. Space **`O(n²)` cho cây** (tới `~ 4n²/3` nút ở trường hợp xấu nhất) và `O(log n)` ngăn xếp đệ quy.

**Nút thắt:** ô nào đó bị **quét lại ở mọi tầng** (từ gốc xuống), dù ta chỉ muốn biết "vùng này đồng nhất không". Nếu trả lời câu hỏi đó trong `O(1)`, ta bỏ được hệ số `log n`.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai hướng cải tiến, đều xuất phát từ việc **bỏ quét lặp**:

1. **Hỏi nhanh bằng bảng:** tổng tiền tố 2D cho biết tổng vùng trong `O(1)`; "toàn 0" ⇔ tổng `0`, "toàn 1" ⇔ tổng `size²`. (Cách 2.)
2. **Đừng hỏi — hãy suy ra từ con:** dựng bốn con trước. Nếu **cả bốn con đều là lá cùng giá trị** thì vùng này đồng nhất ⇒ thay bốn lá bằng **một lá**; ngược lại giữ nút không lá. Không còn quét ô nào! (Cách 3 và Cách 4, khác nhau ở chỗ đệ quy từ trên xuống hay lặp từng tầng từ dưới lên.)

Cả hai đều cho **`O(n²)`**: số nút của cây ở mức tệ nhất cũng chỉ tỉ lệ với số ô, và mỗi nút tốn `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Chia để trị + tổng tiền tố 2D (kiểm tra vùng trong `O(1)`)

**Giải thích từng đoạn.**

1. Dựng `S = build_prefix(grid)` một lần: `O(n²)`.
2. `dfs(r, c, size)`: tính `total = region_sum(S, r, c, size)`.
3. Nếu `total == 0` → lá `False`; nếu `total == size*size` → lá `True`.
4. Ngược lại chia 4 góc, đệ quy như Cách 1.

In [9]:
def construct_prefix(grid):
    n = len(grid)
    S = build_prefix(grid)

    def dfs(r, c, size):
        total = region_sum(S, r, c, size)
        if total == 0:
            return Node(False, True, None, None, None, None)
        if total == size * size:
            return Node(True, True, None, None, None, None)
        h = size // 2
        return Node(True, False,
                    dfs(r, c, h),
                    dfs(r, c + h, h),
                    dfs(r + h, c, h),
                    dfs(r + h, c + h, h))

    return dfs(0, 0, n)


print(serialize(construct_prefix(grid_ex1)) == exp_ex1, serialize(construct_prefix(grid_ex2)) == exp_ex2)

True True


In [10]:
class Solution2:
    def construct(self, grid: list) -> 'Node':
        n = len(grid)
        S = [[0] * (n + 1) for _ in range(n + 1)]
        for i in range(n):
            for j in range(n):
                S[i + 1][j + 1] = S[i][j + 1] + S[i + 1][j] - S[i][j] + grid[i][j]

        def dfs(r, c, size):
            total = S[r + size][c + size] - S[r][c + size] - S[r + size][c] + S[r][c]
            if total == 0:
                return Node(False, True, None, None, None, None)
            if total == size * size:
                return Node(True, True, None, None, None, None)
            h = size // 2
            return Node(True, False,
                        dfs(r, c, h), dfs(r, c + h, h),
                        dfs(r + h, c, h), dfs(r + h, c + h, h))

        return dfs(0, 0, n)


print(check_quadtree(Solution2().construct(grid_ex2), grid_ex2))

(True, 'ok')


**Độ phức tạp (Cách 2).** Dựng bảng `O(n²)`; mỗi nút của cây xử lý `O(1)` và số nút `O(n²)` ⇒ Time **`O(n²)`**. Space **`O(n²)`** (bảng tiền tố + cây).

### Cách 3 — Dựng từ dưới lên: đệ quy, rồi **gộp** nếu 4 con là lá giống nhau

**Giải thích từng đoạn.**

1. Trường hợp cơ sở `size == 1`: trả về lá `Node(bool(grid[r][c]), True, ...)`.
2. Dựng **bốn con** bằng đệ quy (đã là cây tối giản).
3. Nếu **cả bốn con là lá** và **cùng `val`** → vùng đồng nhất → trả **một lá** (bỏ bốn con).
4. Ngược lại → nút không lá với 4 con.

Không quét ô nào: thông tin "đồng nhất" đi **từ dưới lên**.

In [11]:
def construct_merge(grid):
    def dfs(r, c, size):
        if size == 1:
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        tl, tr = dfs(r, c, h), dfs(r, c + h, h)
        bl, br = dfs(r + h, c, h), dfs(r + h, c + h, h)
        kids = (tl, tr, bl, br)
        if all(k.isLeaf for k in kids) and len({k.val for k in kids}) == 1:
            return Node(tl.val, True, None, None, None, None)       # gộp 4 lá giống nhau
        return Node(True, False, tl, tr, bl, br)

    return dfs(0, 0, len(grid))


print(serialize(construct_merge(grid_ex1)) == exp_ex1, serialize(construct_merge(grid_ex2)) == exp_ex2)

True True


In [12]:
class Solution3:
    def construct(self, grid: list) -> 'Node':
        def dfs(r, c, size):
            if size == 1:
                return Node(bool(grid[r][c]), True, None, None, None, None)
            h = size // 2
            tl, tr = dfs(r, c, h), dfs(r, c + h, h)
            bl, br = dfs(r + h, c, h), dfs(r + h, c + h, h)
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf \
                    and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)
            return Node(True, False, tl, tr, bl, br)

        return dfs(0, 0, len(grid))


print(check_quadtree(Solution3().construct(grid_ex2), grid_ex2))

(True, 'ok')


**Độ phức tạp (Cách 3).** `T(n) = 4·T(n/2) + O(1)` ⇒ **`O(n²)`** (cộng số nút theo tầng: `1 + 4 + 16 + ... + n²`). Space **`O(log n)`** ngăn xếp, **không tính** cây trả về (`O(n²)`). Đây là cách gọn và nhanh nhất cho code nộp.

### Cách 4 — Gộp **từng tầng** bằng vòng lặp (không đệ quy)

Nhìn ngược: bắt đầu với `n × n` **lá** (mỗi ô một lá). Lặp: gom **từng khối `2×2`** các nút thành một nút của tầng trên (gộp lại nếu bốn nút là lá cùng giá trị). Mỗi lần lặp cạnh giảm một nửa: `n → n/2 → ... → 1`. Khi còn đúng một nút → đó là gốc.

**Giải thích từng đoạn.**

1. `layer[i][j] = Node(bool(grid[i][j]), True, ...)`.
2. `while len(layer) > 1`: tạo `nxt` cạnh `len(layer)//2`; với mỗi `(i, j)`, lấy `layer[2i][2j]` (trên-trái), `layer[2i][2j+1]` (trên-phải), `layer[2i+1][2j]` (dưới-trái), `layer[2i+1][2j+1]` (dưới-phải).
3. Gộp hoặc tạo nút không lá như Cách 3. `layer = nxt`.
4. Trả `layer[0][0]`.

In [13]:
def merge4(tl, tr, bl, br):
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        return Node(tl.val, True, None, None, None, None)
    return Node(True, False, tl, tr, bl, br)


def construct_layers(grid):
    n = len(grid)
    layer = [[Node(bool(grid[i][j]), True, None, None, None, None) for j in range(n)] for i in range(n)]
    while len(layer) > 1:
        m = len(layer) // 2
        layer = [[merge4(layer[2 * i][2 * j], layer[2 * i][2 * j + 1],
                         layer[2 * i + 1][2 * j], layer[2 * i + 1][2 * j + 1])
                  for j in range(m)] for i in range(m)]
    return layer[0][0]


print(serialize(construct_layers(grid_ex1)) == exp_ex1, serialize(construct_layers(grid_ex2)) == exp_ex2)

True True


In [14]:
class Solution4:
    def construct(self, grid: list) -> 'Node':
        def merge4(tl, tr, bl, br):
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf \
                    and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)
            return Node(True, False, tl, tr, bl, br)

        n = len(grid)
        layer = [[Node(bool(v), True, None, None, None, None) for v in row] for row in grid]
        while len(layer) > 1:
            m = len(layer) // 2
            layer = [[merge4(layer[2 * i][2 * j], layer[2 * i][2 * j + 1],
                             layer[2 * i + 1][2 * j], layer[2 * i + 1][2 * j + 1])
                      for j in range(m)] for i in range(m)]
        return layer[0][0]


print(check_quadtree(Solution4().construct(grid_ex2), grid_ex2))

(True, 'ok')


**Độ phức tạp (Cách 4).** Tầng đáy `n²` lá, tầng kế `(n/2)²`, ... ⇒ tổng `O(n²)`. Time **`O(n²)`**. Space **`O(n²)`** (bảng `layer` đáy `n×n` nút). Ưu điểm: không đệ quy, độ sâu không là vấn đề (ở đây độ sâu chỉ `≤ 6` nên đệ quy cũng an toàn).

## 8. TRACE từng bước

Trace **Cách 1** (quét vùng) trên lưới `4×4` ở mục 4: in ra từng vùng `(r, c, size)`, kết quả kiểm tra đồng nhất, và quyết định.

In [15]:
g4 = [[1, 1, 0, 0],
      [1, 1, 0, 1],
      [0, 0, 1, 1],
      [0, 0, 1, 1]]

def show_quad(node, name="gốc", indent=0):
    pad = "  " * indent
    if node.isLeaf:
        print(f"{pad}{name}: LÁ val={int(node.val)}")
        return
    print(f"{pad}{name}: không lá")
    for nm, k in (("TL", node.topLeft), ("TR", node.topRight), ("BL", node.bottomLeft), ("BR", node.bottomRight)):
        show_quad(k, nm, indent + 1)


def trace_construct(grid):
    def dfs(r, c, size, depth=0):
        pad = "  " * depth
        cells = [grid[i][j] for i in range(r, r + size) for j in range(c, c + size)]
        if len(set(cells)) == 1:
            print(f"{pad}vùng (r={r}, c={c}, size={size}) các ô={cells} -> đồng nhất -> LÁ {cells[0]}")
            return Node(bool(cells[0]), True, None, None, None, None)
        print(f"{pad}vùng (r={r}, c={c}, size={size}) các ô={cells} -> lẫn lộn -> chia 4")
        h = size // 2
        kids = [dfs(r, c, h, depth + 1), dfs(r, c + h, h, depth + 1),
                dfs(r + h, c, h, depth + 1), dfs(r + h, c + h, h, depth + 1)]
        return Node(True, False, *kids)

    return dfs(0, 0, len(grid))


tree4 = trace_construct(g4)
show_quad(tree4)
print(serialize(tree4))
print(check_quadtree(tree4, g4), "số nút:", count_nodes(tree4))

vùng (r=0, c=0, size=4) các ô=[1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1] -> lẫn lộn -> chia 4
  vùng (r=0, c=0, size=2) các ô=[1, 1, 1, 1] -> đồng nhất -> LÁ 1
  vùng (r=0, c=2, size=2) các ô=[0, 0, 0, 1] -> lẫn lộn -> chia 4
    vùng (r=0, c=2, size=1) các ô=[0] -> đồng nhất -> LÁ 0
    vùng (r=0, c=3, size=1) các ô=[0] -> đồng nhất -> LÁ 0
    vùng (r=1, c=2, size=1) các ô=[0] -> đồng nhất -> LÁ 0
    vùng (r=1, c=3, size=1) các ô=[1] -> đồng nhất -> LÁ 1
  vùng (r=2, c=0, size=2) các ô=[0, 0, 0, 0] -> đồng nhất -> LÁ 0
  vùng (r=2, c=2, size=2) các ô=[1, 1, 1, 1] -> đồng nhất -> LÁ 1
gốc: không lá
  TL: LÁ val=1
  TR: không lá
    TL: LÁ val=0
    TR: LÁ val=0
    BL: LÁ val=0
    BR: LÁ val=1
  BL: LÁ val=0
  BR: LÁ val=1
[[0, 1], [1, 1], [0, 1], [1, 0], [1, 1], None, None, None, None, [1, 0], [1, 0], [1, 0], [1, 1]]
(True, 'ok') số nút: 9


Bảng trạng thái (thứ tự thăm tiền thứ tự):

| Vùng `(r,c,size)` | Các ô | Đồng nhất? | Hành động |
|---|---|---|---|
| (0,0,4) | cả lưới | không | chia 4 |
| (0,0,2) | `1,1,1,1` | có | lá `1` |
| (0,2,2) | `0,0,0,1` | không | chia 4 |
| (0,2,1) | `0` | có | lá `0` |
| (0,3,1) | `0` | có | lá `0` |
| (1,2,1) | `0` | có | lá `0` |
| (1,3,1) | `1` | có | lá `1` |
| (2,0,2) | `0,0,0,0` | có | lá `0` |
| (2,2,2) | `1,1,1,1` | có | lá `1` |

Tổng `9` nút, khớp tính tay ở mục 4. Giờ trace **Cách 3** (gộp từ dưới lên) cho lưới toàn `1` cạnh `4` để thấy hiện tượng **gộp**:

In [16]:
def trace_merge(grid):
    def dfs(r, c, size, depth=0):
        pad = "  " * depth
        if size == 1:
            print(f"{pad}ô ({r},{c}) -> lá {grid[r][c]}")
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        tl, tr = dfs(r, c, h, depth + 1), dfs(r, c + h, h, depth + 1)
        bl, br = dfs(r + h, c, h, depth + 1), dfs(r + h, c + h, h, depth + 1)
        kids = (tl, tr, bl, br)
        same = all(k.isLeaf for k in kids) and len({k.val for k in kids}) == 1
        print(f"{pad}vùng (r={r},c={c},size={size}): 4 con là lá cùng giá trị? {same}")
        if same:
            return Node(tl.val, True, None, None, None, None)
        return Node(True, False, tl, tr, bl, br)

    return dfs(0, 0, len(grid))


all_ones = [[1] * 4 for _ in range(4)]
t = trace_merge(all_ones)
print("kết quả: một lá?", t.isLeaf, "| serialize =", serialize(t))

    ô (0,0) -> lá 1
    ô (0,1) -> lá 1
    ô (1,0) -> lá 1
    ô (1,1) -> lá 1
  vùng (r=0,c=0,size=2): 4 con là lá cùng giá trị? True
    ô (0,2) -> lá 1
    ô (0,3) -> lá 1
    ô (1,2) -> lá 1
    ô (1,3) -> lá 1
  vùng (r=0,c=2,size=2): 4 con là lá cùng giá trị? True
    ô (2,0) -> lá 1
    ô (2,1) -> lá 1
    ô (3,0) -> lá 1
    ô (3,1) -> lá 1
  vùng (r=2,c=0,size=2): 4 con là lá cùng giá trị? True
    ô (2,2) -> lá 1
    ô (2,3) -> lá 1
    ô (3,2) -> lá 1
    ô (3,3) -> lá 1
  vùng (r=2,c=2,size=2): 4 con là lá cùng giá trị? True
vùng (r=0,c=0,size=4): 4 con là lá cùng giá trị? True
kết quả: một lá? True | serialize = [[1, 1]]


## 9. Code nộp LeetCode

Bản nộp (Cách 3). Trên LeetCode **không** khai báo lại `Node`, và luôn truyền đủ 6 tham số khi tạo nút.

In [17]:
class Solution:
    def construct(self, grid: list[list[int]]) -> 'Node':
        def dfs(r, c, size):
            if size == 1:
                return Node(bool(grid[r][c]), True, None, None, None, None)
            h = size // 2
            tl = dfs(r, c, h)
            tr = dfs(r, c + h, h)
            bl = dfs(r + h, c, h)
            br = dfs(r + h, c + h, h)
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf \
                    and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)
            return Node(True, False, tl, tr, bl, br)

        return dfs(0, 0, len(grid))


assert serialize(Solution().construct(grid_ex1)) == exp_ex1
assert serialize(Solution().construct(grid_ex2)) == exp_ex2
print("hai ví dụ của đề: OK")

hai ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**Cách 1 / 2 (từ trên xuống).** Đúng theo định nghĩa: vùng đồng nhất → lá; ngược lại chia 4. Vùng `size = 1` luôn đồng nhất nên đệ quy dừng. Số vùng con luôn nguyên vì `size` là luỹ thừa của 2.

**Cách 3 (từ dưới lên).** Quy nạp theo `size`. Với `size = 1` đúng hiển nhiên. Giả sử bốn con (cạnh `size/2`) đều là **cây tối giản** của vùng tương ứng. Vùng cha đồng nhất ⇔ cả bốn vùng con đồng nhất **và** cùng giá trị ⇔ (do tính tối giản) cả bốn con là **lá cùng `val`**. Đó chính xác là điều kiện ta gộp. Khi gộp ta nhận lá đúng như đề; khi không gộp, vùng cha **không** đồng nhất, nên theo đề phải là nút không lá với bốn con — chính là bốn cây tối giản đã có. Vậy kết quả trùng với cây định nghĩa bởi đề (cây tối giản là **duy nhất** cho mỗi lưới, trừ `val` của nút không lá).

**Cách 4** làm cùng phép gộp theo từng tầng nên đúng với cùng lập luận.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space (ngoài cây kết quả) |
|---|---|---|
| 1. Quét vùng | `O(n² log n)` | `O(log n)` |
| 2. Prefix sum | `O(n²)` | `O(n²)` |
| 3. Dựng rồi gộp (đệ quy) | `O(n²)` | `O(log n)` |
| 4. Gộp từng tầng | `O(n²)` | `O(n²)` |

`n` là cạnh lưới (`n ≤ 64`, nên `n² ≤ 4096`; mọi cách đều chạy tức thì). Cây kết quả luôn có tối đa `(4n² − 1)/3` nút (khi mọi ô lẻ loi, mọi vùng đều bị chia đến `1×1`).

## 12. Bộ test

Hàm kiểm tra: `check_quadtree(root, grid)` — dựng lại lưới, kiểm tra cấu trúc, kiểm tra tối giản. Hàm so sánh cây: ta còn đối chiếu **dạng chuẩn** `canon` (bỏ qua `val` của nút không lá) giữa các cách.

In [18]:
import random

def canon(node):
    if node.isLeaf:
        return ("L", int(node.val))
    return ("N", canon(node.topLeft), canon(node.topRight), canon(node.bottomLeft), canon(node.bottomRight))

ALL = {
    "cach1": construct_scan,
    "cach2": construct_prefix,
    "cach3": construct_merge,
    "cach4": construct_layers,
    "nop":   lambda g: Solution().construct(g),
}

def run_all(grid):
    results = {}
    for name, fn in ALL.items():
        t = fn(grid)
        ok, msg = check_quadtree(t, grid)
        assert ok, (name, grid, msg)
        results[name] = canon(t)
    assert len(set(results.values())) == 1, grid
    return results["cach3"]

for g in (grid_ex1, grid_ex2, g4):
    run_all(g)
print("ví dụ đề + lưới 4x4: OK")

ví dụ đề + lưới 4x4: OK


### 12.1. Các trường hợp biên

In [19]:
# n = 1: một lá
for v in (0, 1):
    t = Solution().construct([[v]])
    assert t.isLeaf and t.val == bool(v) and t.topLeft is None
    assert serialize(t) == [[1, v]]
    run_all([[v]])
print("n=1: OK")

# toàn 0 / toàn 1 với mọi n: một lá duy nhất
for n in (1, 2, 4, 8, 16, 32, 64):
    for v in (0, 1):
        t = Solution().construct([[v] * n for _ in range(n)])
        assert t.isLeaf and count_nodes(t) == 1
print("lưới đồng nhất: một lá: OK")

# bàn cờ: mọi ô đều khác ô kề -> cây đầy đủ không gộp được
for n in (2, 4, 8):
    chess = [[(i + j) % 2 for j in range(n)] for i in range(n)]
    t = Solution().construct(chess)
    run_all(chess)
    assert count_nodes(t) == (4 * n * n - 1) // 3, (n, count_nodes(t))
print("bàn cờ: số nút = (4n^2 - 1)/3: OK")

# Chỉ một ô khác biệt ở mỗi vị trí trong lưới 8x8
for i in range(8):
    for j in range(8):
        g = [[0] * 8 for _ in range(8)]
        g[i][j] = 1
        run_all(g)
print("một ô lẻ ở mọi vị trí: OK")

n=1: OK
lưới đồng nhất: một lá: OK
bàn cờ: số nút = (4n^2 - 1)/3: OK
một ô lẻ ở mọi vị trí: OK


### 12.2. Phản ví dụ cho các cách làm sai

Ba hàm sai điển hình (giải thích ở mục 13); giám khảo phải **bắt được** chúng:

In [20]:
def wrong_no_offset(grid):                 # quên cộng offset (r, c) khi chia góc
    def dfs(r, c, size):
        if uniform(grid, r, c, size):
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        return Node(True, False, dfs(r, c, h), dfs(0, h, h), dfs(h, 0, h), dfs(h, h, h))
    return dfs(0, 0, len(grid))

def wrong_corners_only(grid):              # chỉ so sánh 4 góc của vùng
    def dfs(r, c, size):
        s = size - 1
        if len({grid[r][c], grid[r][c + s], grid[r + s][c], grid[r + s][c + s]}) == 1:
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        return Node(True, False, dfs(r, c, h), dfs(r, c + h, h), dfs(r + h, c, h), dfs(r + h, c + h, h))
    return dfs(0, 0, len(grid))

def wrong_no_merge(grid):                  # dựng từ dưới lên nhưng QUÊN gộp
    def dfs(r, c, size):
        if size == 1:
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        return Node(True, False, dfs(r, c, h), dfs(r, c + h, h), dfs(r + h, c, h), dfs(r + h, c + h, h))
    return dfs(0, 0, len(grid))


bad_grid = [[1, 1, 0, 0], [1, 1, 0, 1], [0, 0, 1, 1], [0, 0, 1, 1]]
corner_trap = [[1, 0, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 0, 1]]    # 4 góc đều 1 nhưng có 0 ở giữa
for fn, g in ((wrong_no_offset, bad_grid), (wrong_corners_only, corner_trap), (wrong_no_merge, bad_grid)):
    ok, msg = check_quadtree(fn(g), g)
    print(fn.__name__, "->", ok, "|", msg)
    assert not ok

wrong_no_offset -> False | lưới dựng lại khác lưới gốc
wrong_corners_only -> False | lưới dựng lại khác lưới gốc
wrong_no_merge -> False | chưa gộp: 4 lá cùng giá trị dưới một nút không lá


### 12.3. Đối chiếu bốn cách trên lưới ngẫu nhiên (seed cố định)

Hai kiểu lưới: **nhiễu thuần** (mỗi ô `0/1` ngẫu nhiên, nhiều lá nhỏ) và **theo khối** (chia đệ quy ngẫu nhiên rồi tô đều, để có nhiều vùng đồng nhất lớn — kiểu lưới mà quad tree thật sự có ích).

In [21]:
def gen_noise(n, p):
    return [[1 if random.random() < p else 0 for _ in range(n)] for _ in range(n)]

def gen_blocky(n, p_split=0.6):
    grid = [[0] * n for _ in range(n)]

    def fill(r, c, size):
        if size == 1 or random.random() > p_split:
            v = random.randint(0, 1)
            for i in range(r, r + size):
                for j in range(c, c + size):
                    grid[i][j] = v
            return
        h = size // 2
        for dr, dc in ((0, 0), (0, h), (h, 0), (h, h)):
            fill(r + dr, c + dc, h)

    fill(0, 0, n)
    return grid

random.seed(427)
cnt = 0
for _ in range(300):
    n = random.choice([1, 2, 4, 8, 16, 32, 64])
    run_all(gen_noise(n, random.choice([0.05, 0.5, 0.95])))
    run_all(gen_blocky(n))
    cnt += 2
print("đã đối chiếu", cnt, "lưới ngẫu nhiên: OK")

đã đối chiếu 600 lưới ngẫu nhiên: OK


### 12.4. Cây thật sự nén được

Với lưới theo khối, số nút của cây phải **nhỏ hơn nhiều** so với `(4n² − 1)/3` (nút tối đa). Kiểm tra tính chất này bằng thống kê:

In [22]:
random.seed(5)
n = 64
full = (4 * n * n - 1) // 3
sizes = [count_nodes(Solution().construct(gen_blocky(n, 0.55))) for _ in range(30)]
print("số nút tối đa:", full, "| nhỏ nhất/lớn nhất trong 30 lưới khối:", min(sizes), max(sizes))
assert max(sizes) < full

số nút tối đa: 5461 | nhỏ nhất/lớn nhất trong 30 lưới khối: 1 597


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên cộng offset khi chia góc

In [23]:
g = [[1, 1, 0, 0], [1, 1, 0, 1], [0, 0, 1, 1], [0, 0, 1, 1]]
t = wrong_no_offset(g)
print(check_quadtree(t, g))
print("lưới dựng lại:")
for row in decode_grid(t, 4):
    print(row)

(False, 'lưới dựng lại khác lưới gốc')
lưới dựng lại:
[1, 1, 0, 1]
[1, 1, 1, 1]
[0, 0, 1, 1]
[0, 0, 1, 1]


**Vì sao sai:** góc phần tư "trên-phải" của vùng `(r, c, size)` bắt đầu ở `(r, c + h)`, không phải `(0, h)`. Bản sai luôn nhìn vào **góc của cả lưới** nên vùng con ở nơi khác bị đọc nhầm dữ liệu. **Đúng:** luôn truyền `r + (0 hoặc h)`, `c + (0 hoặc h)`.

### Lỗi 2: Chỉ kiểm tra vài ô thay vì cả vùng

In [24]:
trap = [[1, 0, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 0, 1]]
t = wrong_corners_only(trap)
print("gốc là lá?", t.isLeaf, "(sai: lưới có cả 0 lẫn 1)")
print(check_quadtree(t, trap))

gốc là lá? True (sai: lưới có cả 0 lẫn 1)
(False, 'lưới dựng lại khác lưới gốc')


**Vì sao sai:** bốn góc cùng bằng `1` không có nghĩa cả vùng toàn `1`. Phải xét **mọi** ô (hoặc dùng tổng tiền tố). **Đúng:** `all(grid[i][j] == first ...)` trên cả vùng.

### Lỗi 3: Dựng từ dưới lên nhưng quên gộp

In [25]:
t = wrong_no_merge([[1, 1], [1, 1]])
print("lưới toàn 1 cạnh 2 ->", serialize(t))           # 5 nút thay vì 1
print(check_quadtree(t, [[1, 1], [1, 1]]))
print("đúng phải là:", serialize(Solution().construct([[1, 1], [1, 1]])))

lưới toàn 1 cạnh 2 -> [[0, 1], [1, 1], [1, 1], [1, 1], [1, 1]]
(False, 'chưa gộp: 4 lá cùng giá trị dưới một nút không lá')
đúng phải là: [[1, 1]]


**Vì sao sai:** đề nói vùng đồng nhất phải là **một lá**. Dựng đủ 4 lá con rồi "để đó" cho ra cây không tối giản (đáp án khác). **Đúng:** sau khi dựng 4 con, nếu cả bốn là lá cùng `val` thì trả **một lá**.

### Lỗi 4: Thiếu trường hợp cơ sở (hoặc chia khi `size = 1`)

In [26]:
def wrong_base_case(grid):
    def dfs(r, c, size):
        h = size // 2                       # size = 1 -> h = 0 -> vùng cạnh 0 -> chia mãi
        if size == 0:
            return None
        tl = dfs(r, c, h)
        return Node(True, False, tl, None, None, None)
    return dfs(0, 0, len(grid))

try:
    wrong_base_case([[1]])
    print("không lỗi (nhưng cây sai)")
except Exception as e:
    print(type(e).__name__, e)
t = wrong_base_case([[1]])
print("lưới [[1]] ->", serialize(t), "(đáng lẽ là [[1, 1]])")

không lỗi (nhưng cây sai)
lưới [[1]] -> [[0, 1]] (đáng lẽ là [[1, 1]])


**Vì sao sai:** khi `size = 1`, `h = 0`; vùng con cạnh `0` không còn ô nào. Một ô luôn đồng nhất nên phải dừng ở **lá**. **Đúng:** `if size == 1: return lá` (Cách 3) hoặc `if uniform(...)` đứng trước khi chia (Cách 1).

## 14. Mẹo & nhận dạng pattern

- **"Chia vùng thành 4 phần bằng nhau, đệ quy" ⇒ chia để trị trên lưới.** Hãy nghĩ `(r, c, size)` thay vì cắt lưới (copy tốn thời gian).
- **Kiểm tra vùng đồng nhất lặp đi lặp lại ⇒ tổng tiền tố 2D** (`O(1)` mỗi truy vấn).
- **"Thông tin đồng nhất" có thể suy ra từ con** — mẫu gộp từ dưới lên (tương tự *Balanced Binary Tree 110*, *Symmetric Tree*): hàm trả về một giá trị đủ để cha quyết định.
- Bài trả **cây/đối tượng**: tự viết **hàm tuần tự hóa** để so sánh/in và **hàm giải mã** (dựng lại dữ liệu gốc) để đối chiếu — đây là cách kiểm tra độc lập mạnh nhất.
- Kích thước là **luỹ thừa của 2** ⇒ chia đôi luôn đều, độ sâu chỉ `log₂ n`.
- Luôn truyền **đủ mọi tham số** khi tạo `Node` ở LeetCode (tránh phụ thuộc giá trị mặc định).

## 15. Biến thể

1. **Quad Tree Intersection (558):** cho hai quad tree của hai lưới, trả quad tree của phép **OR** hai lưới.
2. **Dựng lại lưới từ quad tree** (đảo ngược bài này) — chính là `decode_grid` ở mục 3.6.
3. **Lưới `n` bất kỳ (không phải luỹ thừa 2):** chia thành 4 phần không đều (`h = (size + 1) // 2`) và cần xử lý phần rỗng.
4. **Cây bát phân (octree)** cho khối 3D `n×n×n`: chia 8.
5. **Đếm số lá / số nút / độ sâu** của quad tree; **đếm ô `1`** mà không dựng lại lưới.
6. **Hình vuông lớn nhất toàn `1` (221 Maximal Square)** dùng ý tưởng tương tự: dựa vào ba ô lân cận (dp) hoặc prefix sum để hỏi "vùng này toàn 1 không".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space (phụ) | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét vùng | kiểm tra từng ô, chia nếu chưa đồng nhất | `O(n² log n)` | `O(log n)` | hiểu đề, oracle |
| 2. Prefix sum | tổng vùng `O(1)` để biết đồng nhất | `O(n²)` | `O(n²)` | khi cần hỏi đồng nhất nhiều lần |
| 3. Dựng rồi gộp | 4 con là lá cùng giá trị ⇒ gộp | `O(n²)` | `O(log n)` | **nên nộp** (gọn, nhanh) |
| 4. Gộp từng tầng | bắt đầu từ `n²` lá, gom khối `2×2` | `O(n²)` | `O(n²)` | thích vòng lặp, không đệ quy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `serialize(Solution().construct([[1, 1], [1, 1]]))`; (b) `serialize(Solution().construct([[1, 0], [0, 1]]))`; (c) `count_nodes` của lưới toàn `0` cạnh `8`; (d) `region_sum` của vùng `(0, 0, 2)` trong lưới `[[1,1],[1,0]]` (dùng `build_prefix`).

**Bài 2 (tính tay).** Với lưới `4×4`:

```
0 0 1 1
0 0 1 1
1 1 0 1
1 1 1 1
```
hãy vẽ quad tree, tính số nút và viết `serialize` (theo quy tắc mục 3.5).

**Bài 3 (điền TODO).** Hoàn thành `construct_todo(grid)` theo Cách 3.

In [27]:
def construct_todo(grid):
    def dfs(r, c, size):
        # TODO 1: nếu size == 1 thì trả về lá Node(bool(grid[r][c]), True, None, None, None, None)
        # TODO 2: h = size // 2
        # TODO 3: dựng 4 con tl, tr, bl, br bằng dfs với đúng offset (r, c)
        # TODO 4: nếu cả 4 con là lá và cùng val thì trả về một lá có val đó
        # TODO 5: ngược lại trả về Node(True, False, tl, tr, bl, br)
        pass

    # TODO 6: gọi dfs(0, 0, len(grid)) và trả về kết quả
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `my_decode(root, n)` dựng lại lưới `n×n` từ quad tree (không xem `decode_grid`); thử `my_decode(Solution().construct(g), len(g)) == g` trên vài lưới.

**Bài 5 (biến thể / mini challenge).** (a) **Quad Tree Intersection (558):** viết `intersect(a, b)` trả về quad tree của `grid_a OR grid_b` (gợi ý: nếu `a` là lá: `1` ⇒ trả lá `1`, `0` ⇒ trả `b`; ngược lại với `b`; còn lại đệ quy 4 góc rồi gộp). Kiểm tra bằng cách OR hai lưới rồi `check_quadtree`. (b) Viết `count_ones(root, n)` đếm số ô `1` của lưới **mà không dựng lại lưới** (mỗi lá `1` ở vùng cạnh `s` đóng góp `s*s`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: lưới đồng nhất ⇒ **một lá** `[[1, 1]]`; bàn cờ `2×2` ⇒ `[[0,1],[1,1],[1,0],[1,0],[1,1]]`; lưới toàn `0` ⇒ `1` nút; tổng vùng `(0,0,2)` là `3`. Bài 3: nhớ `size == 1` trước.
- **Mức 2 (vừa):** Bài 2: góc trên-trái `0` (lá), trên-phải `1` (lá), dưới-trái `1` (lá), dưới-phải `[[0,1],[1,1]]` **không đồng nhất** ⇒ chia 4 lá `0,1,1,1`. Vậy `1 + 4 + 4 = 9` nút. Bài 4: đệ quy `fill(node, r, c, size)` đổ giá trị cho lá cả vùng.
- **Mức 3 (sâu):** Bài 5(a): nếu `a.isLeaf`: `return Node(True, True, ...)` khi `a.val` còn lại `return b`; tương tự `b`. Sau khi đệ quy 4 góc, **gộp** nếu cả 4 là lá cùng `val`. 5(b): đệ quy: `if node.isLeaf: return s*s if node.val else 0`; nếu không thì cộng 4 con với `s//2`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert serialize(Solution().construct([[1, 1], [1, 1]])) == [[1, 1]]
assert serialize(Solution().construct([[1, 0], [0, 1]])) == [[0, 1], [1, 1], [1, 0], [1, 0], [1, 1]]
assert count_nodes(Solution().construct([[0] * 8 for _ in range(8)])) == 1
assert region_sum(build_prefix([[1, 1], [1, 0]]), 0, 0, 2) == 3

# Bài 2
g2 = [[0, 0, 1, 1], [0, 0, 1, 1], [1, 1, 0, 1], [1, 1, 1, 1]]
t2 = Solution().construct(g2)
assert count_nodes(t2) == 9
assert serialize(t2) == [[0, 1], [1, 0], [1, 1], [1, 1], [0, 1]] + [None] * 12 + [
                         [1, 0], [1, 1], [1, 1], [1, 1]]
assert check_quadtree(t2, g2) == (True, "ok")


# Bài 3
def construct_todo_answer(grid):
    def dfs(r, c, size):
        if size == 1:
            return Node(bool(grid[r][c]), True, None, None, None, None)
        h = size // 2
        tl, tr = dfs(r, c, h), dfs(r, c + h, h)
        bl, br = dfs(r + h, c, h), dfs(r + h, c + h, h)
        if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
            return Node(tl.val, True, None, None, None, None)
        return Node(True, False, tl, tr, bl, br)

    return dfs(0, 0, len(grid))

assert canon(construct_todo_answer(grid_ex2)) == canon(Solution().construct(grid_ex2))


# Bài 4
def my_decode(root, n):
    grid = [[0] * n for _ in range(n)]

    def fill(node, r, c, size):
        if node.isLeaf:
            for i in range(r, r + size):
                for j in range(c, c + size):
                    grid[i][j] = 1 if node.val else 0
            return
        h = size // 2
        fill(node.topLeft, r, c, h)
        fill(node.topRight, r, c + h, h)
        fill(node.bottomLeft, r + h, c, h)
        fill(node.bottomRight, r + h, c + h, h)

    fill(root, 0, 0, n)
    return grid

for g in (grid_ex1, grid_ex2, g2, [[1]]):
    assert my_decode(Solution().construct(g), len(g)) == g


# Bài 5
def intersect(a, b):
    if a.isLeaf:
        return a if a.val else b
    if b.isLeaf:
        return b if b.val else a
    tl, tr = intersect(a.topLeft, b.topLeft), intersect(a.topRight, b.topRight)
    bl, br = intersect(a.bottomLeft, b.bottomLeft), intersect(a.bottomRight, b.bottomRight)
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        return Node(tl.val, True, None, None, None, None)
    return Node(True, False, tl, tr, bl, br)

random.seed(55)
for _ in range(100):
    n = random.choice([1, 2, 4, 8, 16])
    ga, gb = gen_blocky(n), gen_blocky(n)
    union = [[ga[i][j] | gb[i][j] for j in range(n)] for i in range(n)]
    res = intersect(Solution().construct(ga), Solution().construct(gb))
    assert check_quadtree(res, union) == (True, "ok")


def count_ones(node, n):
    if node.isLeaf:
        return n * n if node.val else 0
    h = n // 2
    return sum(count_ones(k, h) for k in (node.topLeft, node.topRight, node.bottomLeft, node.bottomRight))

for _ in range(50):
    n = random.choice([1, 2, 4, 8, 16])
    g = gen_blocky(n)
    assert count_ones(Solution().construct(g), n) == sum(map(sum, g))
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đệ quy từ dưới lên rồi gộp)

```python
class Solution:
    def construct(self, grid: List[List[int]]) -> 'Node':
        def dfs(r, c, size):
            if size == 1:
                return Node(bool(grid[r][c]), True, None, None, None, None)
            h = size // 2
            tl = dfs(r, c, h)
            tr = dfs(r, c + h, h)
            bl = dfs(r + h, c, h)
            br = dfs(r + h, c + h, h)
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf \
                    and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)
            return Node(True, False, tl, tr, bl, br)

        return dfs(0, 0, len(grid))
```

**Checklist**

- [ ] Vùng được biểu diễn bằng `(r, c, size)` và **mọi** lời gọi con đều cộng đúng offset (`c + h`, `r + h`)?
- [ ] Có trường hợp cơ sở `size == 1` (hoặc kiểm tra đồng nhất **trước** khi chia)?
- [ ] "Đồng nhất" kiểm tra **cả vùng** (hoặc suy ra từ 4 lá con), không chỉ vài ô?
- [ ] Vùng đồng nhất cho **một lá**, không để lại bốn lá con (cây tối giản)?
- [ ] Lá có đủ bốn con `None`; nút không lá có đủ bốn con?
- [ ] Truyền đủ 6 tham số khi tạo `Node`; không khai báo lại `Node` trong bài nộp?
- [ ] Đã thử `n = 1`, lưới toàn 0/toàn 1, bàn cờ, ví dụ 8×8 của đề; đã so sánh `canon` giữa các cách?